# Deep Learning-Based PCB Defect Inspection
**RV College of Engineering — AI Foundations for Engineers [CI124TA]**

**Instructions:** Runtime → Change runtime type → **T4 GPU** → Run All

In [ ]:
# ── 1. GPU check ──────────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
    print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 2. Clone / update project + clear stale .pyc cache ────────────────────────
import os, sys

PROJECT = '/content/pcb-error-detection'
if os.path.isdir(PROJECT):
    !git -C {PROJECT} pull
else:
    !git clone https://github.com/jacklachan/pcb-error-detection.git {PROJECT}

# Wipe compiled bytecode so Python always reads the latest .py source
!find {PROJECT} -name '*.pyc' -delete
!find {PROJECT} -name '__pycache__' -type d -exec rm -rf {{}} + 2>/dev/null; echo 'cache cleared'

!pip install -q scipy tqdm

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

# Remove any stale module objects from a previous run
for _m in list(sys.modules.keys()):
    if _m.startswith(('data', 'models', 'utils')):
        del sys.modules[_m]

print('Done.')

In [ ]:
# ── 3. Download DeepPCB dataset ────────────────────────────────────────────────
import os, glob

DATA_ROOT = '/content/DeepPCB'

# Always start fresh — stale partial clones cause the "0 pairs" bug
if os.path.isdir(DATA_ROOT):
    existing = glob.glob(os.path.join(DATA_ROOT,'**','*.jpg'), recursive=True)
    if len(existing) < 100:
        print(f'Stale directory ({len(existing)} images) — wiping and re-cloning...')
        !rm -rf {DATA_ROOT}

if not os.path.isdir(DATA_ROOT):
    print('Cloning DeepPCB (~380 MB, may take 3-5 min)...')
    !git clone https://github.com/tangsanli5201/DeepPCB.git {DATA_ROOT}

# Verify what we got
all_imgs = glob.glob(os.path.join(DATA_ROOT,'**','*.jpg'), recursive=True)
all_txts = glob.glob(os.path.join(DATA_ROOT,'**','*.txt'), recursive=True)
print(f'\nImages : {len(all_imgs)}')
print(f'Txt    : {len(all_txts)}')
print('\n── First 10 image paths ──────────────────────────────────────────')
for p in sorted(all_imgs)[:10]: print(' ', p)
print('\n── trainval.txt sample ───────────────────────────────────────────')
tv = os.path.join(DATA_ROOT,'PCBData','trainval.txt')
if os.path.exists(tv):
    with open(tv) as f:
        for line in f.readlines()[:5]: print(' ', line.strip())
else:
    print('  trainval.txt NOT FOUND')

In [ ]:
# ── 4. Load dataset ────────────────────────────────────────────────────────────
import importlib, os
import data.dataset as _ds_mod
importlib.reload(_ds_mod)
from data.dataset import DeepPCBDataset, _find_pairs, DEFECT_CLASSES
from data.transforms import PCBTransform
import numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
from collections import Counter

_MEAN  = np.array([0.485, 0.456, 0.406])
_STD   = np.array([0.229, 0.224, 0.225])
COLORS = ['#FF4444','#44DD44','#4488FF','#FFDD00','#FF44FF','#44FFFF']
def denorm(t):
    return np.clip(t.permute(1,2,0).numpy()*_STD+_MEAN, 0, 1)

print('Scanning for pairs...')
all_pairs = _find_pairs(DATA_ROOT)
print(f'Pairs found: {len(all_pairs)}')
if all_pairs:
    print('Sample triple:')
    for p in all_pairs[:2]: print(' ', p)

train_ds = DeepPCBDataset(DATA_ROOT, split='train',
                          transform=PCBTransform((512,512), train=False))
val_ds   = DeepPCBDataset(DATA_ROOT, split='test',
                          transform=PCBTransform((512,512), train=False))
print(f'\nTrain: {len(train_ds)}  |  Val: {len(val_ds)}')

DATA_ROOT_DS = DATA_ROOT

counts = Counter()
for _,_,t in train_ds:
    counts.update(t['labels'].tolist())
fig, ax = plt.subplots(figsize=(8,3))
ax.bar([DEFECT_CLASSES[i] for i in range(6)], [counts[i] for i in range(6)], color=COLORS)
ax.set_title('DeepPCB — class distribution (train)')
ax.set_ylabel('Count'); plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
# ── 5. Visualise sample pairs ─────────────────────────────────────────────────
if len(train_ds) == 0:
    print('train_ds is empty — skipping visualisation.')
else:
    n_show = min(3, len(train_ds))
    idxs   = np.random.choice(len(train_ds), n_show, replace=False)
    fig, axes = plt.subplots(n_show, 2, figsize=(14, 4*n_show))
    if n_show == 1: axes = [axes]
    for row, idx in enumerate(idxs):
        tmpl, test, target = train_ds[int(idx)]
        boxes, labels = target['boxes'].numpy(), target['labels'].numpy()
        axes[row][0].imshow(denorm(tmpl)); axes[row][0].set_title('Template',fontsize=10); axes[row][0].axis('off')
        axes[row][1].imshow(denorm(test)); axes[row][1].set_title(f'Test — {len(boxes)} defect(s)',fontsize=10); axes[row][1].axis('off')
        for box, lbl in zip(boxes, labels):
            cx,cy,bw,bh = box; x1,y1=(cx-bw/2)*512,(cy-bh/2)*512
            axes[row][1].add_patch(patches.Rectangle((x1,y1),bw*512,bh*512,
                linewidth=2,edgecolor=COLORS[lbl],facecolor='none'))
            axes[row][1].text(x1,y1-4,DEFECT_CLASSES[lbl],color=COLORS[lbl],fontsize=8,fontweight='bold')
    plt.suptitle('Sample PCB pairs with ground-truth annotations',fontsize=13)
    plt.tight_layout(); plt.show()

In [ ]:
# ── 6. Model summary ──────────────────────────────────────────────────────────
from models.pcb_detector import PCBDefectDetector

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

model = PCBDefectDetector(backbone='resnet50', num_queries=100, num_classes=6, pretrained=True)
sections = {'Backbone':model.backbone,'Fusion C2':model.fusion_c2,'Fusion C3':model.fusion_c3,
            'Fusion C4 (attn)':model.fusion_c4,'Fusion C5 (attn)':model.fusion_c5,
            'FPN':model.fpn,'Detection Head':model.head}
print(f'\n{"Module":<24} {"Params":>10}')
print('-'*36)
for name, mod in sections.items():
    print(f'{name:<24} {count_params(mod)/1e6:>8.2f}M')
print('-'*36)
print(f'{"TOTAL":<24} {count_params(model)/1e6:>8.2f}M')

In [ ]:
# ── 7. Training configuration ─────────────────────────────────────────────────
import os
CONFIG = dict(
    data_root=DATA_ROOT, img_size=512, batch_size=4,
    num_queries=100, decoder_layers=6, epochs=100,
    lr=1e-4, weight_decay=1e-4, backbone='resnet50',
    output_dir='/content/output',
)
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print('Config:', CONFIG)

In [ ]:
# ── 8. Train ──────────────────────────────────────────────────────────────────
import torch
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from tqdm.notebook import tqdm
from IPython.display import clear_output
from utils.losses import PCBDetectionLoss

device  = torch.device('cuda')
use_amp = True

def collate_fn(batch):
    t, s, tgt = zip(*batch)
    return torch.stack(t), torch.stack(s), list(tgt)

# DATA_ROOT_DS is set in cell 4 (real dataset or synthetic fallback)
train_ds2 = DeepPCBDataset(DATA_ROOT_DS, split='train',
    transform=PCBTransform((CONFIG['img_size'],)*2, train=True))
val_ds2   = DeepPCBDataset(DATA_ROOT_DS, split='test',
    transform=PCBTransform((CONFIG['img_size'],)*2, train=False))
print(f'Train: {len(train_ds2)}  |  Val: {len(val_ds2)}')

train_loader = DataLoader(train_ds2, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=2, collate_fn=collate_fn, pin_memory=True)
val_loader   = DataLoader(val_ds2, batch_size=1,
    shuffle=False, num_workers=2, collate_fn=collate_fn)

model = PCBDefectDetector(
    backbone=CONFIG['backbone'], num_queries=CONFIG['num_queries'],
    num_classes=6, pretrained=True, num_decoder_layers=CONFIG['decoder_layers'],
).to(device)

criterion = PCBDetectionLoss(num_classes=6).to(device)
param_dicts = [
    {'params': [p for n,p in model.named_parameters() if 'backbone' not in n and p.requires_grad]},
    {'params': [p for n,p in model.named_parameters() if 'backbone' in n and p.requires_grad],
     'lr': CONFIG['lr']*0.1},
]
optimizer = torch.optim.AdamW(param_dicts, lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
scaler    = GradScaler(enabled=use_amp)

train_losses, val_losses = [], []
best_val = float('inf')

for epoch in range(1, CONFIG['epochs']+1):
    model.train(); t_loss = 0
    for tmpl, test, tgts in tqdm(train_loader, desc=f'Epoch {epoch}/{CONFIG["epochs"]}', leave=False):
        tmpl = tmpl.to(device, non_blocking=True)
        test = test.to(device, non_blocking=True)
        tgts = [{k: v.to(device) for k,v in t.items()} for t in tgts]
        with autocast(enabled=use_amp):
            losses = criterion(model(tmpl, test), tgts)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        scaler.step(optimizer); scaler.update()
        t_loss += losses['total'].item()
    scheduler.step()

    model.eval(); v_loss = 0
    with torch.no_grad():
        for tmpl, test, tgts in tqdm(val_loader, desc='val', leave=False):
            tmpl = tmpl.to(device); test = test.to(device)
            tgts = [{k: v.to(device) for k,v in t.items()} for t in tgts]
            with autocast(enabled=use_amp):
                losses = criterion(model(tmpl, test), tgts)
            v_loss += losses['total'].item()

    t_avg = t_loss/len(train_loader); v_avg = v_loss/len(val_loader)
    train_losses.append(t_avg); val_losses.append(v_avg)
    print(f'Epoch {epoch:3d}/{CONFIG["epochs"]}  Train={t_avg:.4f}  Val={v_avg:.4f}')

    if v_avg < best_val:
        best_val = v_avg
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_loss': best_val, 'args': CONFIG},
                   f'{CONFIG["output_dir"]}/best_model.pth')
        print(f'  ✓ Saved (val={best_val:.4f})')

    if epoch % 5 == 0:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9,4))
        ax.plot(train_losses, label='Train'); ax.plot(val_losses, label='Val')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title(f'Training — epoch {epoch}/{CONFIG["epochs"]}')
        ax.legend(); ax.grid(True); plt.tight_layout(); plt.show()

print('Training complete. Best val loss:', round(best_val, 4))

In [ ]:
# ── 9. Evaluate — mAP@0.5 + P/R/F1 ───────────────────────────────────────────
import os
ckpt_path = f'{CONFIG["output_dir"]}/best_model.pth'
if not os.path.exists(ckpt_path):
    print('No checkpoint found — training may still be running or failed. Skipping evaluation.')
else:
    from utils.metrics import compute_map, DEFECT_CLASSES
    from utils.box_ops import box_cxcywh_to_xyxy, box_iou
    from tqdm.notebook import tqdm

    ckpt = torch.load(ckpt_path)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for tmpl, test, tgts in tqdm(val_loader, desc='Evaluating'):
            out = model(tmpl.to(device), test.to(device))
            for i, tgt in enumerate(tgts):
                logits = out['pred_logits'][i].softmax(-1)
                scores, labels = logits[:,:-1].max(-1)
                keep = scores > 0.05
                all_preds.append({'boxes':out['pred_boxes'][i][keep].cpu(),
                                  'labels':labels[keep].cpu(),'scores':scores[keep].cpu()})
                all_targets.append({'boxes':tgt['boxes'],'labels':tgt['labels']})

    per_ap, map_val = compute_map(all_preds, all_targets, iou_threshold=0.5)
    print(f'\nmAP@0.5 = {map_val:.4f}\n')
    for cls, ap in per_ap.items():
        bar = '█'*int(30*(ap if ap==ap else 0))
        print(f'  {cls:<20} {ap:.4f if ap==ap else "N/A":>6}  {bar}')

    tp=[0]*6; fp=[0]*6; fn=[0]*6
    for preds, tgts in zip(all_preds, all_targets):
        keep = preds['scores']>0.5
        pb=box_cxcywh_to_xyxy(preds['boxes'][keep]); pl=preds['labels'][keep]
        gb=box_cxcywh_to_xyxy(tgts['boxes']); gl=tgts['labels']
        matched=torch.zeros(len(gb),dtype=torch.bool)
        for b,l in zip(pb,pl):
            hit=False
            if len(gb):
                ious,_=box_iou(b.unsqueeze(0),gb); best_iou,best_j=ious[0].max(0)
                if best_iou>.5 and gl[best_j]==l and not matched[best_j]:
                    tp[l.item()]+=1; matched[best_j]=True; hit=True
            if not hit: fp[l.item()]+=1
        for j,gl_j in enumerate(gl):
            if not matched[j]: fn[gl_j.item()]+=1
    print(f'\n{"Class":<20} {"Prec":>8} {"Rec":>8} {"F1":>8}'); print('-'*48)
    for c,name in enumerate(DEFECT_CLASSES):
        p=tp[c]/(tp[c]+fp[c]+1e-8); r=tp[c]/(tp[c]+fn[c]+1e-8); f=2*p*r/(p+r+1e-8)
        print(f'{name:<20} {p:>8.4f} {r:>8.4f} {f:>8.4f}')
    stp,sfp,sfn=sum(tp),sum(fp),sum(fn)
    p=stp/(stp+sfp+1e-8); r=stp/(stp+sfn+1e-8); f=2*p*r/(p+r+1e-8)
    print('-'*48); print(f'{"Overall":<20} {p:>8.4f} {r:>8.4f} {f:>8.4f}')

In [ ]:
# ── 10. Visualise predictions ─────────────────────────────────────────────────
import os
if 'val_ds2' not in dir() or len(val_ds2) == 0:
    print('val_ds2 not available — run training cell first.')
elif not os.path.exists(f'{CONFIG["output_dir"]}/best_model.pth'):
    print('No checkpoint yet — training may still be running.')
else:
    from utils.box_ops import box_cxcywh_to_xyxy
    n_show = min(4, len(val_ds2))
    idxs   = np.random.choice(len(val_ds2), n_show, replace=False)
    fig, axes = plt.subplots(n_show, 2, figsize=(14, 4*n_show))
    if n_show == 1: axes = [axes]
    for row, idx in enumerate(idxs):
        tmpl_t, test_t, target = val_ds2[int(idx)]
        with torch.no_grad():
            out = model(tmpl_t.unsqueeze(0).to(device), test_t.unsqueeze(0).to(device))
        logits = out['pred_logits'][0].softmax(-1)
        scores, labels = logits[:,:-1].max(-1); keep = scores > 0.5
        axes[row][0].imshow(denorm(tmpl_t)); axes[row][0].set_title('Template',fontsize=9); axes[row][0].axis('off')
        ax = axes[row][1]; ax.imshow(denorm(test_t))
        ax.set_title(f'Predictions ({keep.sum().item()} detected)',fontsize=9); ax.axis('off')
        for box,lbl,sc in zip(box_cxcywh_to_xyxy(out['pred_boxes'][0][keep]).cpu().numpy()*512,
                              labels[keep].cpu().numpy(), scores[keep].cpu().numpy()):
            x1,y1,x2,y2=box
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor=COLORS[lbl],facecolor='none'))
            ax.text(x1,y1-4,f'{DEFECT_CLASSES[lbl]} {sc:.2f}',color=COLORS[lbl],fontsize=7,fontweight='bold')
        for box in box_cxcywh_to_xyxy(target['boxes']).numpy()*512:
            x1,y1,x2,y2=box
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=1.5,edgecolor='white',facecolor='none',linestyle='--'))
    plt.suptitle('Solid=predictions  Dashed white=ground truth',fontsize=12)
    plt.tight_layout(); plt.show()

In [ ]:
# ── 11. Download checkpoint ───────────────────────────────────────────────────
import os
ckpt_path = f'{CONFIG["output_dir"]}/best_model.pth'
if os.path.exists(ckpt_path):
    from google.colab import files
    files.download(ckpt_path)
    print('Checkpoint downloaded.')
else:
    print('No checkpoint yet — training must complete first.')